In [ ]:
# ==============================================================================
# File: 04_Launch_Live_Demo.ipynb (Cell 1 of 3)
# What this notebook does: Mounts Google Drive, clones/syncs https://github.com/Alekhagrawal/ZeroFake.git,
#   installs all required dependencies, syncs saved evaluation CSVs/plots from Google Drive,
#   and launches the minimalist ZeroFake Gradio web dashboard (app.py) with a public *.gradio.live share URL.
# Inputs: /content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake/best_model.pt.
# Outputs: Live public *.gradio.live URL for classroom presentation and interactive testing.
# ==============================================================================

# --- CELL 1: Mount Google Drive, Pull Latest GitHub Repo & Install Dependencies ---
import os
import shutil
import sys
from google.colab import drive

# 1. Safely clear any unmounted placeholder /content/drive directory before mounting
if not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')

# 2. Clone or force-sync https://github.com/Alekhagrawal/ZeroFake.git into /content/ZeroFake
GITHUB_REPO_URL = 'https://github.com/Alekhagrawal/ZeroFake.git'
REPO_DIR = '/content/ZeroFake'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f'[INFO] Fetching and syncing latest commits from {GITHUB_REPO_URL}...')
    !git -C /content/ZeroFake fetch origin
    !git -C /content/ZeroFake reset --hard origin/main
else:
    print(f'[INFO] Cloning {GITHUB_REPO_URL} into {REPO_DIR}...')
    !rm -rf /content/ZeroFake
    !git clone https://github.com/Alekhagrawal/ZeroFake.git /content/ZeroFake

%cd /content/ZeroFake
os.environ['PYTHONPATH'] = '/content/ZeroFake'
if '/content/ZeroFake' not in sys.path:
    sys.path.insert(0, '/content/ZeroFake')

# 3. Install facenet-pytorch with --no-deps first (avoiding Pillow 10.2.0 compilation on Python 3.13)
#    and install remaining packages using 'grad-cam'
!pip install -q --no-deps facenet-pytorch
!pip install -q timm opencv-python-headless scikit-learn matplotlib seaborn grad-cam gradio pyyaml tqdm

# 4. Pre-flight CPU dry-run check on app.py
!PYTHONPATH=/content/ZeroFake python -u app.py --dry-run
if _exit_code != 0:
    raise RuntimeError(f'app.py --dry-run failed with exit code {_exit_code}')
print('[SUCCESS] Google Drive mounted, repo synced at /content/ZeroFake, and app.py --dry-run verified!')


In [ ]:
# --- CELL 2: Verify Trained ZeroFake Checkpoint & Sync Saved Evaluation Results ---
# Plain-English explanation:
#   Verifies that your trained Full ZeroFake checkpoint exists on Google Drive and copies
#   any saved evaluation CSVs and 300-DPI PNG charts from Google Drive into /content/ZeroFake/results/
#   so the 'Model Benchmarks & Evaluation Results' dropdown in the web app is populated.

import os
import shutil

STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
CKPT_PATH = os.path.join(STORAGE_DIR, 'checkpoints', 'zerofake', 'best_model.pt')
DRIVE_RESULTS_DIR = os.path.join(STORAGE_DIR, 'results')
LOCAL_RESULTS_DIR = '/content/ZeroFake/results'

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Trained ZeroFake checkpoint not found at: {CKPT_PATH}')

if os.path.exists(DRIVE_RESULTS_DIR):
    shutil.copytree(DRIVE_RESULTS_DIR, LOCAL_RESULTS_DIR, dirs_exist_ok=True)
    print(f'[SUCCESS] Synced evaluation tables and plots from {DRIVE_RESULTS_DIR} to {LOCAL_RESULTS_DIR}.')

print(f'[READY] Verified trained checkpoint: {CKPT_PATH}')


In [ ]:
# --- CELL 3: Launch Live ZeroFake Minimalist Web App (Public *.gradio.live Share Link) ---
# Plain-English explanation:
#   Launches app.py with share=True, generating a live public https://*.gradio.live link.
#   Simply drop or paste any face image (or upload a short video clip) for instant 3-way
#   open-set detection, and expand the dropdown accordions below the verdict card during your viva.

%cd /content/ZeroFake
!PYTHONPATH=/content/ZeroFake python -u app.py --checkpoint /content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake/best_model.pt --results_dir /content/ZeroFake/results
